# Linear Combination of Unitaries, Block Encoding, and Qubitization

Quantum computing processes information by encoding it in quantum states and applying unitary operations to them. This means that for any operation you want to apply to your state, you need to be able to implement it as a unitary transformation. This is easy (or at least straightforward) if this operation is a unitary already. But what do you do if the operation you're interested in is not unitary?

Block encoding is the method used to encode a non-unitary matrix as a "block" inside a larger unitary matrix. It allows us to apply the non-unitary operation to the target quantum register _probabilistically_ by applying the larger unitary matrix to a larger quantum register composed of the target register and some auxiliary qubits.

In this kata, we will go through the principles of block encoding for matrices represented as a linear combination of unitaries and learn to implement it for several simple example scenarios. We will also explore _qubitization_ - a technique used to convert block encoding to a slightly different format that turns out to be more useful when block encoding serves as an input to other algorithms such as quantum phase estimation.

**This kata covers the following topics:**

- Linear combination of unitaries, in particular Pauli matrices
- Block encoding
- Qubitization
- Testing the results of block encoding and qubitization

## Linear combination of unitaries (LCU)

As the name implies, for the LCU method we write the non-unitary matrix $A$ as a linear combination of several unitaries $U_k$ with positive, real-valued coefficients:

$$
A = \sum_{k=0}^{K-1} \alpha_k U_k
$$

A common choice for $U_k$ is the set of Pauli matrices $I, X, Y, Z$ for representing $2 \times 2$ matrices or their tensor products for larger sizes. Since the Pauli matrices or their tensor products are a basis for matrices of corresponding size, such representation is always possible. (In fact, some methods of converting problem descriptions into Hamiltonian matrices express them in the Pauli basis naturally.)

> If the coefficients $\alpha_k$ in the linear combination have to be negative or complex-valued, we can always modify our choice of the unitaries to absorb the sign and/or the complex phase into them and keep the coefficients real and positive.

## Block encoding: The protocol

The block encoding protocol consists of two main components:

1. PREPARE - a state preparation unitary that acts on an auxiliary register to prepare a superposition of $K$ basis states with the amplitudes $\sqrt{\alpha_k / \lambda}$, where $\lambda = \sum_k \alpha_k$ is the normalization coefficient.  

2. SELECT - a self-inverse unitary that maps each of the basis states in the auxiliary register to one of the unitary terms in the LCU and applies them to the target register conditionally.

To implement the block encoding protocol, these unitaries are combined as follows:

1. PREPARE transforms the auxiliary register to the superposition state defined by the amplitudes $\sqrt{\alpha_k / \lambda}$.
2. SELECT applies the unitaries $U_k$ to the target register based on the values of the auxiliary register.
3. UnPREPARE - the adjoint of the PREPARE uncomputes the auxiliary register.

The matrix unitary transformation composed of these steps contains the original matrix $A$ as its top left block (possibly scaled by a constant factor to enable this embedding). In other words, it is exactly what block encoding should do!

Let's walk through several examples of using block encoding for matrices of simple form and see what the results look like.

## Example 1: $2 \times 2$ diagonal matrix

To start with, we'll consider a very simple example: a $2 \times 2$ diagonal real-valued matrix $A$ that can be expressed as a sum of just two Pauli matrices: $I$ and $Z$:

$$
A = \begin{pmatrix}
        \beta_0 & 0 \\
        0 & \beta_1
    \end{pmatrix}
= \underbrace{\frac12(\beta_0 + \beta_1)}_{\alpha_0} \underbrace{\vphantom{\frac12} I}_{U_0} + \underbrace{\frac12(\beta_0 - \beta_1)}_{\alpha_1} \underbrace{\vphantom{\frac12} Z}_{U_1}
$$

We'll say that $\beta_0 > \beta_1 > 0$ to keep our coefficients $\alpha_k$ positive.

Let's implement the PREPARE and SELECT unitaries for this example and then see how they work together to block-encode the matrix $A$.

### Problem 1.1. Prepare a single-qubit state

In our example, we have a linear combination of $K=2$ unitaries, so we only need an auxiliary register of size $1$ to prepare a superposition of $\ket{0}$ and $\ket{1}$ in it. The amplitudes of this superposition are defined by the values $\alpha_0$ and $\alpha_1$.

**Inputs:** 

1. A `Qubits` register of length $1$ in the $\ket{0}$ state.
2. Real positive numbers $\alpha_0$ and $\alpha_1$, represented as `list[float]`.

**Goal:** Transform the qubit into the state $\tfrac{1}{\sqrt{\alpha_0 + \alpha_1}} \left(\sqrt{\alpha_0} \ket{0} + \sqrt{\alpha_1} \ket{1} \right)$.

In [ ]:
from psiqdk.workbench import Qubits
from test_LCUBlockEncoding import problem

@problem
class OneQubitPrepare(Qubrick):
    def _compute(self, reg: Qubits, alpha: list[float]) -> None:
        # Write your code here
        ...

### Problem 1.2. Select between I and Z

**Inputs:** 

1. A `Qubits` register `index` of length $1$ in an arbitrary superposition state.
2. A `Qubits` register `target` of length $1$ in an arbitrary superposition state.

**Goal:**

Apply the SELECT unitary to these two registers:

* If `index` is in the $\ket{0}$ state, apply $U_0 = I$ to the `target` register (in other words, do nothing).
* If `index` is in the $\ket{1}$ state, apply $U_1 = Z$ to the `target` register.
* The effect of the unitary if `index` is in a superposition state is defined by its linearity.

In [ ]:
from psiqdk.workbench import Qubits
from test_LCUBlockEncoding import problem

@problem
class OneQubitSelectIZ(Qubrick):
    def _compute(self, index: Qubits, target: Qubits) -> None:
        # Write your code here
        ...

### Demo 1: Block encoding for $2 \times 2$ diagonal matrices

The following code shows how implement the block encoding protocol as a Qubrick and how to extract the unitary matrix of the transformation is does. You can see that, indeed, the top left $2 \times 2$ block of the matrix is the matrix $A$. In this case, it is not even scaled, just embedded as it is!

In [ ]:
from math import atan2, sqrt
from psiqdk.workbench import QPU, Qubits, Qubrick, units
from test_LCUBlockEncoding import print_highlighted_matrix

class OneQubitPrepare(Qubrick):
    def _compute(self, reg: Qubits, alpha: list[float]) -> None:
        theta = 2 * atan2(sqrt(alpha[1]), sqrt(alpha[0]))
        reg.ry(theta * units.rad)

class OneQubitSelectIZ(Qubrick):
    def _compute(self, index: Qubits, target: Qubits) -> None:
        target.z(cond=index)

class BlockEncoding(Qubrick):
    def __init__(self, prepare: Qubrick, select: Qubrick, **kwargs) -> None:
        self.prepare = prepare
        self.select = select
        super().__init__(**kwargs)

    def _compute(self, index: Qubits, target: Qubits, alpha: list[float], verbose: bool = False) -> None:
        if verbose:
            print("State before block encoding:")
            self.qc.print_state_vector()
        with self.prepare.computed(index, alpha):
            if verbose:
                print("State after PREPARE:")
                self.qc.print_state_vector()
            self.select.compute(index, target)
            if verbose:
                print("State after SELECT:")
                self.qc.print_state_vector()
        if verbose:
            print("State after UnPREPARE:")
            self.qc.print_state_vector()


# The values on the diagonal of the matrix A
beta = [1, 0.28]
A = [[beta[0], 0], [0, beta[1]]]
print_highlighted_matrix(A, 2, "A")

# The corresponding coefficients of the linear combination of unitaries
alpha = [(beta[0] + beta[1]) / 2, (beta[0] - beta[1]) / 2]
print(f"{alpha=}")

# Initialize the QPU
qpu = QPU(num_qubits=2, pre_filters=[">>unitary>>"])

# Allocate the registers
target = Qubits(1, "target", qpu)
index = Qubits(1, "index", qpu)

# Instantiate and apply the Qubrick
block_encoding = BlockEncoding(OneQubitPrepare(), OneQubitSelectIZ())
block_encoding.compute(index, target, alpha)

# Get the matrix of the unitary implemented by the Qubrick
ufilter = qpu.get_filter_by_name('>>unitary>>')
full_matrix = ufilter.get()
print_highlighted_matrix(full_matrix.real, 2, "Full Matrix")

1,0
0,0.28


alpha=[0.64, 0.36]


1.0,0.0,0.0,0.0
0.0,0.28,0.0,-0.96
0.0,0.0,1.0,0.0
0.0,-0.96,0.0,-0.28


### Math 1: The evolution of the quantum state

What does it mean to implement a unitary that contains a specific matrix as a top left block?

Let's trace each step involved in the block encoding protocol for this example and see how the quantum state evolves during it. We'll follow the math performed on each step and validate it using the code.

We'll use the same example we've used in the demo above:

$$
A = \begin{pmatrix}
        1 & 0 \\
        0 & 0.28
    \end{pmatrix} = 0.64 I + 0.36 Z
$$

#### Initial state and goal state

We'll start with the target qubit in the following state:

$$\ket{\psi_0} = 0.6\ket{0} + 0.8\ket{1}$$

Our goal is for it to end up in the state 

$$\ket{\psi_{goal}} = A \ket{\psi_0} = 1 \cdot 0.6\ket{0} + 0.28 \cdot 0.8\ket{1} = 0.6\ket{0} + 0.224 \ket{1}$$

Since $A$ is not a unitary matrix, this goal state is not normalized and thus is not a valid quantum state! 

To work around this, we will modify our goal: we'll add an auxiliary ("index") qubit that starts in the $\ket{0}$ state, and we'll aim to transform the joint state of the two qubits as follows:

$$\ket{\psi_0}_{target} \ket{0}_{index} \rightarrow \ket{\psi_{goal}}_{target} \ket{0}_{index} + \ket{\psi_{junk}}_{target} \ket{1}_{index}$$

In other words, we want the target qubit to end up in the state $\ket{\psi_{goal}}$ if the auxiliary qubit ends up in the $\ket{0}$ state (up to a normalization coefficient), and we don't really care what happens to the target qubit if the auxiliary qubit ends up in the $\ket{1}$ state.

#### PREPARE

In our example, $\alpha_0 = 0.64$ and $\alpha_1 = 0.36$. These coefficients are conveniently normalized ($\lambda = \alpha_0 + \alpha_1 = 1$).

The PREPARE unitary for our scenario is a single $Ry$ gate that transforms the auxiliary qubit into a superposition state defined by square roots of the coefficients:

$$\ket{0} \rightarrow \sqrt{0.64} \ket{0} + \sqrt{0.36} \ket{1} = 0.8 \ket{0} + 0.6 \ket{1}$$

> Since we know we're using an $Ry$ gate to implement the PREPARE, we can spell its matrix explicitly:
>
> $$Ry = \begin{bmatrix} 0.8 & -0.6 \\ 0.6 & 0.8 \end{bmatrix}$$
>
> This will come in handy later, when we will be figuring out the form of PREPARE $^\dagger$.

The joint state of the two qubits after this step is

$$(0.6 \ket{0} + 0.8 \ket{1})_{target} (0.8 \ket{0} + 0.6 \ket{1})_{index}$$

$$
= 0.48 \ket{0}_{target} \ket{0}_{index} + 0.36 \ket{0}_{target} \ket{1}_{index} + 0.64 \ket{1}_{target} \ket{0}_{index} + 0.48 \ket{1}_{target} \ket{1}_{index}
$$

#### SELECT

As we've established earlier, the SELECT for our example is simply a controlled $Z$ gate, with the index qubit as the control and the target qubit as the target. After applying SELECT, our system state will change the sign of the $\ket{11}$ term of the superposition:

$$
0.48 \ket{0}_{target} \ket{0}_{index} + 0.36 \ket{0}_{target} \ket{1}_{index} + 0.64 \ket{1}_{target} \ket{0}_{index} - 0.48 \ket{1}_{target} \ket{1}_{index}
$$

$$
= (0.48 \ket{0} + 0.64 \ket{1})_{target} \ket{0}_{index} + (0.36 \ket{0} - 0.48 \ket{1})_{target} \ket{1}_{index}
$$

#### UnPREPARE

Finally, the adjoint of our PREPARE unitary is defined as

$$Ry^\dagger = \begin{bmatrix} 0.8 & 0.6 \\ -0.6 & 0.8 \end{bmatrix}$$

Applying it transforms our state as follows:

$$
(0.48 \ket{0} + 0.64 \ket{1})_{target} Ry^\dagger\ket{0}_{index} + (0.36 \ket{0} - 0.48 \ket{1})_{target} Ry^\dagger\ket{1}_{index}
$$

$$
= (0.48 \ket{0} + 0.64 \ket{1})_{target} (0.8 \ket{0} - 0.6 \ket{1})_{index} + (0.36 \ket{0} - 0.48 \ket{1})_{target} (0.6 \ket{0} + 0.8 \ket{1})_{index}
$$

$$
= (0.48 \cdot 0.8 + 0.36 \cdot 0.6) \ket{0}_{target}\ket{0}_{index}
+ (0.64 \cdot 0.8 - 0.48 \cdot 0.6) \ket{1}_{target}\ket{0}_{index}
$$
$$
+ (-0.48 \cdot 0.6 + 0.36 \cdot 0.8) \ket{0}_{target}\ket{1}_{index}
+ (-0.64 \cdot 0.6 - 0.48 \cdot 0.8) \ket{1}_{target}\ket{1}_{index}
$$

$$
= 0.6 \ket{0}_{target}\ket{0}_{index} + 0.224 \ket{1}_{target}\ket{0}_{index} - 0.768 \ket{1}_{target}\ket{1}_{index}
$$

$$
= \underbrace{(0.6 \ket{0} + 0.224 \ket{1})_{target}}_{\ket{\psi_{goal}}} \ket{0}_{index} \underbrace{- 0.768 \ket{1}_{target}}_{\ket{\psi_{junk}}} \ket{1}_{index}
$$

Conveniently, in this case the final state expression contains the goal state as is, without extra normalization coefficients, which helps us recognize it for what it is.

We can see the same sequence of states printed if we run the block encoding Qubrick in verbose mode:

In [16]:
target.push_state([0.6, 0.8])
block_encoding.compute(index, target, alpha, verbose=True)

State before block encoding:
|target|index>
|0|0>    0.600000+0.000000j
|1|0>    0.800000+0.000000j
State after PREPARE:
|target|index>
|0|0>    0.480000+0.000000j
|0|1>    0.360000+0.000000j
|1|0>    0.640000+0.000000j
|1|1>    0.480000+0.000000j
State after SELECT:
|target|index>
|0|0>    0.480000+0.000000j
|0|1>    0.360000+0.000000j
|1|0>    0.640000+0.000000j
|1|1>    -0.480000-0.000000j
State after UnPREPARE:
|target|index>
|0|0>    0.600000+0.000000j
|1|0>    0.224000+0.000000j
|1|1>    -0.768000-0.000000j


## Example 2: $4 \times 4$ diagonal matrix

Now, let's consider a slightly more complicated example: a $4 \times 4$ diagonal real-valued matrix $A$. This matrix can be expressed as a sum of two-qubit tensor products of Pauli matrices $I \otimes I$, $I \otimes Z$, $Z \otimes I$ and $Z \otimes Z$:

$$
A = \begin{pmatrix}
        \beta_0 & 0 & 0 & 0 \\
        0 & \beta_1 & 0 & 0 \\
        0 & 0 & \beta_2 & 0 \\
        0 & 0 & 0 & \beta_3 
    \end{pmatrix}
= \alpha_0 \underbrace{I \otimes I}_{U_0} + \alpha_1 \underbrace{I \otimes Z}_{U_1} + \alpha_2 \underbrace{Z \otimes I}_{U_2} + \alpha_3 \underbrace{Z \otimes Z}_{U_3}
$$

> Before we begin working out the implementation of this example, let's remind ourselves what tensor products of single-qubit gates look like in Workbench. The tensor product $I \otimes Z$ can be written down as follows:
>
> $$
I \otimes Z =
\begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix} \otimes \begin{bmatrix} 1 & 0 \\ 0 & -1 \end{bmatrix} =
\begin{bmatrix}
    1 & 0 & 0 & 0 \\ 
    0 & -1 & 0 & 0 \\ 
    0 & 0 & 1 & 0 \\ 
    0 & 0 & 0 & -1
\end{bmatrix}$$
>
> In Workbench, this matrix translates to applying the $Z$ gate to the _least significant_ qubit of the two-qubit register, in other words, to qubit $0$. You can confirm this using the following code which does exactly that and prints the matrix of the resulting transformation:

In [20]:
qpu = QPU(num_qubits=2, pre_filters=[">>unitary>>"])
reg = Qubits(2, "reg", qpu)

# Apply the Z gate to the first (least significant) qubit of the register
reg[0].z()

ufilter = qpu.get_filter_by_name('>>unitary>>')
full_matrix = ufilter.get()
print_highlighted_matrix(full_matrix.real, 4, "I₁ ⊗ Z₀")

1.0,0.0,0.0,0.0
0.0,-1.0,0.0,0.0
0.0,0.0,1.0,0.0
0.0,0.0,0.0,-1.0


### Problem 2.1. Find the LCU decomposition

**Input:**
$4$ real positive numbers $\beta_k$, represented as `list[float]` - the diagonal elements of the matrix $A$ above.

**Goal:**
Find the coefficients $\alpha_k$ of the LCU decomposition of this matrix as described above.

In [ ]:
from test_LCUBlockEncoding import problem

@problem
def lcu_decomposition(beta: list[float]) -> list[float]:
    # Write your code here
    ...

### Problem 2.2. Prepare a two-qubit state

This time, we have a linear combination of $K=4$ unitaries, so we need an auxiliary register of size $2$ to prepare a superposition of basis states $\ket{0} ... \ket{3}$ in it. The amplitudes of this superposition are defined by the values $\alpha_k$.

**Inputs:** 

1. A `Qubits` register of length $2$ in the $\ket{0}$ state.
2. $4$ real positive numbers $\alpha_k$, represented as `list[float]`.

**Goal:** Transform the qubit into the state $\tfrac{1}{\sqrt{\alpha_0 + \alpha_1 + \alpha_2 + \alpha_3}} \left(\sqrt{\alpha_0} \ket{0} + \sqrt{\alpha_1} \ket{1} + \sqrt{\alpha_2} \ket{2} + \sqrt{\alpha_3} \ket{3} \right)$.

> You can learn more about preparing arbitrary quantum states in the [Preparing Arbitrary Quantum States](../ArbitraryStatePreparation/ArbitraryStatePreparation.ipynb) kata.

In [ ]:
from psiqdk.workbench import Qubits
from test_LCUBlockEncoding import problem

@problem
class TwoQubitPrepare(Qubrick):
    def _compute(self, reg: Qubits, alpha: list[float]) -> None:
        # Write your code here
        ...

### Problem 2.3. Select between four I and Z tensor products

**Inputs:** 

1. A `Qubits` register `index` of length $2$ in an arbitrary superposition state.
2. A `Qubits` register `target` of length $2$ in an arbitrary superposition state.

**Goal:**

Apply the SELECT unitary to these two registers:

* If `index` is in the $\ket{0}$ state, apply $U_0 = I \otimes I$ to the `target` register (in other words, do nothing).
* If `index` is in the $\ket{1}$ state, apply $U_1 = I \otimes Z$ to the `target` register.
* If `index` is in the $\ket{2}$ state, apply $U_2 = Z \otimes I$ to the `target` register.
* If `index` is in the $\ket{3}$ state, apply $U_3 = Z \otimes Z$ to the `target` register.
* The effect of the unitary if `index` is in a superposition state is defined by its linearity.

In [ ]:
from psiqdk.workbench import Qubits
from test_LCUBlockEncoding import problem

@problem
class TwoQubitSelectIZ(Qubrick):
    def _compute(self, index: Qubits, target: Qubits) -> None:
        # Write your code here
        ...

### Exercise: Block encoding for $4 \times 4$ diagonal matrices

Once we've implemented these two unitaries, we can reuse the block encoding Qubrick from the earlier demo to combine them together. The following code shows how to do this and validates that the $16 \times 16$ matrix of the four-qubit unitary does contain the matrix we're block-encoding in the top left corner.

## Qubitization

## Using block encoding with QPE

# Conclusion

Congratulations! In this kata you learned about block encoding and qubitization and saw how to use these techniques to implement unitaries that act as inputs to quantum phase estimation.
> Copyright (c) 2026 PsiQuantum